# ============================================================
# STEP 3C — KEYWORD-CIRCULARITY ABLATION (TUMC)
# ============================================================
# Addresses the strongest reviewer objection to the ~0.999
# binary AUC: the keyword features (has/num_*_keyword) are
# computed from the SAME lexicons that drove label
# reclassification, so for reclassified URLs they partly
# encode the label by construction.
#
# This script re-runs the honest (Exp C) configuration in four
# feature regimes and reports the AUC/F1 delta attributable to
# the keyword group:
#   R1 full          — all 130 honest features (incl. keywords)
#   R2 no-keywords   — keyword group removed (conservative)
#   R3 keywords-only — ONLY keyword features (upper bound on
#                      how much the circular features alone give)
#   R4 no-graph      — keyword group kept, graph removed
#                      (isolates graph vs keyword contribution)
#
# Runs on the INDUCTIVE feature file (deployment-honest graph).
# Top model only, both tasks, for speed.
#
# Output: step3c_keyword_ablation.csv
# ============================================================

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, f1_score, matthews_corrcoef,
                             recall_score, precision_score)

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive.csv", "features_full_v12.csv"]:
    INPUT_FILE = os.path.join(DATA_DIR, cand)
    if os.path.exists(INPUT_FILE): break
OUT = os.path.join(DATA_DIR, "step3c_keyword_ablation.csv")

TASKS = ["binary", "5class"]
# Always honest: drop the two collection artifacts + 2 leaky graph feats
ARTIFACTS = ["is_tr_domain", "is_https"]
LEAKY_GRAPH = ["cluster_malicious_ratio", "campaign_token_score"]

# Keyword feature group (the circular features)
KEYWORD_FEATS = [
    "has_phishing_keyword","num_phishing_keywords",
    "has_malware_keyword","num_malware_keywords",
    "has_scam_keyword","num_scam_keywords",
    "has_turkish_keyword","num_turkish_keywords",
]
# Graph feature group (the novelty) — for the no-graph regime
GRAPH_FEATS = [
    "rare_token_count","max_token_cluster_size","shared_token_degree",
    "unique_token_ratio","token_reuse_score","domain_family_size",
    "tld_token_cooccur","sibling_domain_count","domain_ngram_cluster",
    "registrant_pattern_score","subdomain_reuse_count","path_template_reuse",
    "host_pattern_degree","campaign_membership","token_centrality","is_hub_domain",
]

import xgboost as xgb
def make_model(is_binary, n_classes):
    if is_binary:
        return xgb.XGBClassifier(
            objective="binary:logistic", eval_metric="logloss",
            max_depth=4, learning_rate=0.03, n_estimators=500,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=1, reg_lambda=2,
            scale_pos_weight=5.5, random_state=SEED, n_jobs=-1)
    return xgb.XGBClassifier(
        objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
        max_depth=4, learning_rate=0.03, n_estimators=500,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=1, reg_lambda=2,
        random_state=SEED, n_jobs=-1)

print("="*70)
print("STEP 3C — KEYWORD-CIRCULARITY ABLATION")
print(f"  Source: {os.path.basename(INPUT_FILE)}")
print("="*70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
honest = [c for c in df.columns if c not in META
          and c not in ARTIFACTS and c not in LEAKY_GRAPH]
present_kw    = [f for f in KEYWORD_FEATS if f in honest]
present_graph = [f for f in GRAPH_FEATS  if f in honest]
print(f"\n  Honest features: {len(honest)}")
print(f"  Keyword features present: {len(present_kw)} → {present_kw}")
print(f"  Graph features present:   {len(present_graph)}")

REGIMES = {
    "R1_full":          honest,
    "R2_no_keywords":   [f for f in honest if f not in present_kw],
    "R3_keywords_only": present_kw,
    "R4_no_graph":      [f for f in honest if f not in present_graph],
}

folds = sorted(df["fold"].unique())
results = []

for task in TASKS:
    target = "label_enc" if task == "binary" else "class_enc"
    is_binary = (task == "binary")
    n_classes = df[target].nunique()
    print(f"\n{'='*60}\n  TASK: {task}\n{'='*60}")
    for rname, feats in REGIMES.items():
        if not feats:
            continue
        t0 = time.time(); fold_m = []
        for tf in folds:
            tr = df[df["fold"] != tf]; te = df[df["fold"] == tf]
            model = make_model(is_binary, n_classes)
            model.fit(tr[feats].fillna(0).values, tr[target].values)
            proba = model.predict_proba(te[feats].fillna(0).values)
            pred  = model.predict(te[feats].fillna(0).values)
            yt = te[target].values
            if is_binary:
                fold_m.append({
                    "auc": roc_auc_score(yt, proba[:,1]),
                    "mcc": matthews_corrcoef(yt, pred),
                    "recall": recall_score(yt, pred, zero_division=0),
                    "precision": precision_score(yt, pred, zero_division=0)})
            else:
                fold_m.append({
                    "f1_macro": f1_score(yt, pred, average="macro", zero_division=0),
                    "mcc": matthews_corrcoef(yt, pred),
                    "auc_ovr": roc_auc_score(yt, proba, multi_class="ovr", average="macro")})
        avg = {k: float(np.mean([m[k] for m in fold_m])) for k in fold_m[0]}
        avg.update({"task":task,"regime":rname,"n_features":len(feats),
                    "time_s":round(time.time()-t0,1)})
        results.append(avg)
        key = "auc" if is_binary else "f1_macro"
        print(f"    {rname:<18s} ({len(feats):>3d}f): {key}={avg[key]:.4f} "
              f"MCC={avg['mcc']:.4f}")

res = pd.DataFrame(results)
res.to_csv(OUT, index=False)

# ── The headline deltas ──────────────────────────────────────
print(f"\n{'='*70}\nKEYWORD-CIRCULARITY VERDICT\n{'='*70}")
for task in TASKS:
    sub = res[res["task"]==task].set_index("regime")
    key = "auc" if task=="binary" else "f1_macro"
    if "R1_full" in sub.index and "R2_no_keywords" in sub.index:
        full = sub.loc["R1_full", key]
        nokw = sub.loc["R2_no_keywords", key]
        drop = full - nokw
        print(f"\n  {task.upper()} ({key}):")
        print(f"    Full (with keywords)  : {full:.4f}")
        print(f"    Without keywords      : {nokw:.4f}")
        print(f"    Keyword contribution  : {drop:+.4f}")
        if "R3_keywords_only" in sub.index:
            print(f"    Keywords alone        : {sub.loc['R3_keywords_only',key]:.4f}")
        if "R4_no_graph" in sub.index:
            print(f"    Without graph         : {sub.loc['R4_no_graph',key]:.4f} "
                  f"(graph contribution {full - sub.loc['R4_no_graph',key]:+.4f})")
        if abs(drop) < 0.005:
            print(f"    → Keyword removal barely changes performance.")
            print(f"      The result does NOT depend on circular features. ✓")
        elif drop < 0.02:
            print(f"    → Modest keyword dependence; report no-keyword as")
            print(f"      the conservative headline number.")
        else:
            print(f"    → Substantial keyword dependence. The no-keyword")
            print(f"      number is the honest one; discuss circularity openly.")

print(f"""
{'='*70}
STEP 3C COMPLETE → {OUT}
{'='*70}
  Report in the paper:
    - R1 vs R2: how much the (circular) keyword features add
    - R4: confirms graph features carry independent signal
    - Use R2 (no-keyword) as the conservative headline if the
      drop is non-trivial; otherwise R1 stands.
{'='*70}
""")

STEP 3C — KEYWORD-CIRCULARITY ABLATION
  Source: features_full_final_inductive.csv

  Honest features: 176
  Keyword features present: 8 → ['has_phishing_keyword', 'num_phishing_keywords', 'has_malware_keyword', 'num_malware_keywords', 'has_scam_keyword', 'num_scam_keywords', 'has_turkish_keyword', 'num_turkish_keywords']
  Graph features present:   16

  TASK: binary
    R1_full            (176f): auc=1.0000 MCC=0.9887
    R2_no_keywords     (168f): auc=1.0000 MCC=0.9887
    R3_keywords_only   (  8f): auc=0.5817 MCC=-0.0002
    R4_no_graph        (160f): auc=1.0000 MCC=0.9888

  TASK: 5class
    R1_full            (176f): f1_macro=0.9055 MCC=0.8684
    R2_no_keywords     (168f): f1_macro=0.7948 MCC=0.8339
    R3_keywords_only   (  8f): f1_macro=0.4965 MCC=0.3727
    R4_no_graph        (160f): f1_macro=0.9055 MCC=0.8683

KEYWORD-CIRCULARITY VERDICT

  BINARY (auc):
    Full (with keywords)  : 1.0000
    Without keywords      : 1.0000
    Keyword contribution  : -0.0000
    Keywords alo